# 02 — Experiment Tracking

## What
Experiment tracking records every training run with its hyperparameters, dataset version, metrics, and artifacts. Without it, teams answer 'what worked?' by trawling through email threads or shared spreadsheets — a losing battle as soon as more than one person trains models.

## Why
Reproducibility is the foundation of scientific credibility and production safety. You cannot responsibly deploy a model you cannot reproduce. MLflow, Weights & Biases, and Neptune are the dominant tools. MLflow is open-source and self-hostable — a good default for teams that cannot send data to third parties.

## Community context
Meta's FBLearner, Google's Vizier, and Microsoft's NNI all emerged from the same pain: engineers re-running experiments already done by colleagues. The 2022 'Reproducibility Crisis in ML' paper found that 74% of NeurIPS experiments could not be reproduced from the published code alone — hyperparameter sensitivity was the primary culprit.

In [ ]:
# Simulated MLflow-style experiment tracking
# (no mlflow install required — we build the tracker from scratch)

import json, time, hashlib, random
from pathlib import Path

class ExperimentTracker:
    def __init__(self, experiment_name, store_dir='/tmp/mlruns'):
        self.experiment_name = experiment_name
        self.store = Path(store_dir) / experiment_name
        self.store.mkdir(parents=True, exist_ok=True)
        self.active_run = None

    def start_run(self, run_name=None):
        run_id = hashlib.md5(str(time.time_ns()).encode()).hexdigest()[:8]
        self.active_run = {
            'run_id': run_id,
            'run_name': run_name or run_id,
            'start_time': time.time(),
            'params': {}, 'metrics': {}, 'tags': {}
        }
        return run_id

    def log_param(self, key, value):
        self.active_run['params'][key] = value

    def log_metric(self, key, value, step=None):
        self.active_run['metrics'].setdefault(key, [])
        self.active_run['metrics'][key].append({'value': value, 'step': step})

    def set_tag(self, key, value):
        self.active_run['tags'][key] = value

    def end_run(self):
        self.active_run['end_time'] = time.time()
        run_file = self.store / (self.active_run['run_id'] + '.json')
        run_file.write_text(json.dumps(self.active_run, indent=2))
        print(f"Run {self.active_run['run_name']} saved to {run_file}")
        result = self.active_run
        self.active_run = None
        return result

    def list_runs(self, order_by='metrics.val_acc', ascending=False):
        runs = []
        for f in self.store.glob('*.json'):
            r = json.loads(f.read_text())
            runs.append(r)
        def get_val(r):
            parts = order_by.split('.')
            v = r
            for p in parts:
                if isinstance(v, dict):
                    v = v.get(p, {})
                elif isinstance(v, list) and v:
                    v = v[-1].get('value', 0)
                else:
                    return 0
            if isinstance(v, list) and v:
                return v[-1]['value']
            return v if isinstance(v, (int, float)) else 0
        runs.sort(key=get_val, reverse=not ascending)
        return runs

# Simulate several training runs with different hyperparameters
tracker = ExperimentTracker('credit_fraud_detection')

configs = [
    {'lr': 0.01, 'depth': 4, 'n_estimators': 100},
    {'lr': 0.05, 'depth': 6, 'n_estimators': 200},
    {'lr': 0.001, 'depth': 3, 'n_estimators': 300},
]

for cfg in configs:
    rid = tracker.start_run(f'lr={cfg["lr"]}_depth={cfg["depth"]}')
    for k, v in cfg.items():
        tracker.log_param(k, v)
    tracker.set_tag('framework', 'xgboost')
    tracker.set_tag('dataset_hash', 'abc123def')
    # Simulate training metrics
    random.seed(hash(str(cfg)))
    base_acc = 0.88 + cfg['depth']*0.01 - cfg['lr']*0.5
    for epoch in range(5):
        acc = base_acc + random.gauss(0, 0.005) + epoch*0.003
        tracker.log_metric('val_acc', round(acc, 4), step=epoch)
        tracker.log_metric('val_auc', round(acc + 0.05, 4), step=epoch)
    tracker.end_run()

print('\n=== Leaderboard (sorted by val_acc) ===')
for r in tracker.list_runs('metrics.val_acc'):
    best_acc = r['metrics']['val_acc'][-1]['value']
    print(f"  {r['run_name']:<35} val_acc={best_acc:.4f}")

## Model Registration

Once the best run is identified, it should be promoted to a model registry — a versioned store that separates *training artifact* from *deployment artifact*. The registry tracks transitions: None → Staging → Production → Archived.

In [ ]:
class ModelRegistry:
    STAGES = ['None', 'Staging', 'Production', 'Archived']

    def __init__(self):
        self.models = {}  # name -> list of versions

    def register(self, model_name, run_id, metrics):
        self.models.setdefault(model_name, [])
        version = len(self.models[model_name]) + 1
        self.models[model_name].append({
            'version': version, 'run_id': run_id,
            'metrics': metrics, 'stage': 'None'
        })
        print(f'Registered {model_name} v{version} from run {run_id}')
        return version

    def transition(self, model_name, version, new_stage):
        assert new_stage in self.STAGES
        entry = self.models[model_name][version-1]
        old = entry['stage']
        entry['stage'] = new_stage
        print(f'{model_name} v{version}: {old} -> {new_stage}')

    def get_production(self, model_name):
        for v in reversed(self.models.get(model_name, [])):
            if v['stage'] == 'Production':
                return v
        return None

registry = ModelRegistry()
runs = tracker.list_runs('metrics.val_acc')
best = runs[0]
v = registry.register('fraud_detector', best['run_id'], best['metrics'])
registry.transition('fraud_detector', v, 'Staging')
registry.transition('fraud_detector', v, 'Production')
prod = registry.get_production('fraud_detector')
print(f'\nProduction model: run {prod["run_id"]}, stage={prod["stage"]}')